# ⏱️ RS-LiDAR & Vanilla LiDAR: Benchmark Table 2 Efficiency (Time & Peak VRAM on A100)

> **Mục tiêu**: Đo đạc độc lập, chuẩn xác thời gian thực thi phân rã 3 thành phần (khớp Bảng 9) và đỉnh VRAM (Bảng 2) của **RS-LiDAR** và **Vanilla LiDAR** trên GPU **NVIDIA A100**:
> 1. **$T_{\text{lookahead}}$**: Thời gian sinh $N$ hạt lookahead (DPM-5 hoặc DMD-1) và giải mã VAE.
> 2. **$T_{\text{reward}}$**: Thời gian đánh giá ImageReward (**Full-Batch 1 forward duy nhất** trên trọn vẹn $M \times N$ ảnh).
> 3. **$T_{\text{target}}$**: Thời gian khử nhiễu mục tiêu (Closed-form steering, sinh 4 ảnh đích, **DỪNG NGAY KHÔNG CHẤM REWARD Ở PHASE 2**).
> 4. **$T_{\text{total}}$**: $T_{\text{lookahead}} + T_{\text{reward}} + T_{\text{target}}$.
> 5. **Peak VRAM**: $\text{torch.cuda.max\_memory\_allocated()} / 1024^3$ (GiB).

---

### 📌 3 Cấu Hình Benchmark Chuẩn Bảng 2:
| Setting Key | Model | Lookahead Solver | Target Solver | Scale | $\eta$ |
| :--- | :--- | :--- | :--- | :---: | :---: |
| `sdv1.5_ddim50` | SD v1.5 | DPM-5 ($N=50$) | DDIM 50 steps ($N=4$) | 12.5 | 0.0 |
| `sdv1.5_ddpm100` | SD v1.5 | DPM-5 ($N=50$) | DDPM 100 steps ($N=4$) | 12.5 | 1.0 |
| `sdxl_dmd1` | SDXL (2.6B) | DMD-1 ($N=100$, 1 bước) | DDPM 100 steps ($N=4$) | 8.0 | 1.0 |

## 1. 🖥️ Kiểm Tra Phần Cứng GPU (Khuyến nghị NVIDIA A100)

In [ ]:
!nvidia-smi

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"\n✅ GPU khả dụng: {gpu_name} ({total_vram:.2f} GiB VRAM)")
    if "A100" in gpu_name:
        print("🌟 Bạn đang chạy trên NVIDIA A100 chuẩn benchmark bài báo ICML 2026!")
    else:
        print("ℹ️ Lưu ý: GPU hiện tại không phải A100. Thời gian thực tế có thể chênh lệch so với Bảng 2 bài báo.")
else:
    print("❌ Không tìm thấy GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU (A100).")

## 2. 📂 Đồng Bộ Mã Nguồn Từ GitHub

In [ ]:
import os, sys

REPO_URL = "https://github.com/leekwanreal/RS-LiDAR.git"
REPO_DIR = "RS-LiDAR"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repo từ {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print("Repo đã tồn tại. Đang cập nhật mã nguồn mới nhất...")
    !cd {REPO_DIR} && git pull origin main

# Chuyển working directory vào Diffusion-LiDAR-Sampling
TARGET_DIR = os.path.join(REPO_DIR, "Diffusion-LiDAR-Sampling")
if os.path.exists(TARGET_DIR):
    os.chdir(TARGET_DIR)
    print(f"\n📂 Thư mục làm việc hiện tại: {os.getcwd()}")
else:
    print(f"\n📂 Thư mục làm việc hiện tại: {os.getcwd()}")

## 3. 📦 Cài Đặt Thư Viện Cần Thiết

In [ ]:
# Cài đặt các gói phụ thuộc tiêu chuẩn
!pip install -q diffusers==0.27.2 transformers==4.40.2 accelerate==0.30.0 ImageReward ftfy timm tabulate pandas

print("\n✅ Cài đặt môi trường hoàn tất!")

## 4. ⚙️ Cấu Hình Thực Nghiệm (Interactive Form)

In [ ]:
# @title 🎛️ Chọn Tham Số Benchmark

# @markdown **1. Cấu hình bài báo (Setting)**:
SETTING = "sdv1.5_ddpm100" # @param ["sdv1.5_ddim50", "sdv1.5_ddpm100", "sdxl_dmd1"]

# @markdown **2. Phương pháp lấy mẫu (Method)**:
METHOD = "rs-lidar" # @param ["rs-lidar", "lidar"]

# @markdown **3. Số lượng prompts đo trung bình**:
NUM_PROMPTS = 3 # @param {type:"integer"}

# @markdown **4. Độ lệch chuẩn nhiễu làm mịn (Sigma)**:
SIGMA = 1.0 # @param {type:"number"}
# @markdown *(Gợi ý: 1.0 cho SD 1.5; 0.5 cho SDXL; Tự động về 0.0 nếu chọn Vanilla LiDAR)*

# @markdown **5. Số mẫu Monte Carlo (M)**:
NUM_MC_SAMPLES = 4 # @param {type:"integer"}
# @markdown *(Gợi ý: 4 cho RS-LiDAR; Tự động về 1 nếu chọn Vanilla LiDAR)*

# @markdown **6. Batch Size ImageReward (Mặc định: Full Batch)**:
REWARD_BATCH_SIZE = "None" # @param ["None", "64", "32", "16"]
# @markdown *(Giữ 'None' để ép chạy trọn vẹn 1 batch duy nhất trên A100!)*

# @markdown **7. Bật Warm-up GPU (1 prompt khởi động kernel)**:
ENABLE_WARMUP = True # @param {type:"boolean"}

print(f"Cấu hình đã chọn:")
print(f"• Setting:           {SETTING}")
print(f"• Method:            {METHOD.upper()}")
print(f"• Prompts to test:   {NUM_PROMPTS}")
print(f"• Sigma:             {SIGMA if METHOD == 'rs-lidar' else 0.0}")
print(f"• Monte Carlo M:     {NUM_MC_SAMPLES if METHOD == 'rs-lidar' else 1}")
print(f"• Reward Batch Size: {REWARD_BATCH_SIZE}")
print(f"• GPU Warmup:        {ENABLE_WARMUP}")

## 5. 🚀 Thực Thi Benchmark Table 2 Efficiency

In [ ]:
import subprocess

warmup_flag = "--warmup" if ENABLE_WARMUP else "--no_warmup"
cmd = [
    "python", "benchmark_table2_efficiency.py",
    "--setting", SETTING,
    "--method", METHOD,
    "--num_prompts", str(NUM_PROMPTS),
    "--sigma", str(SIGMA),
    "--num_mc_samples", str(NUM_MC_SAMPLES),
    "--reward_batch_size", str(REWARD_BATCH_SIZE),
    warmup_flag,
    "--output_dir", "results/benchmark_efficiency"
]

print(f"Chạy lệnh: {' '.join(cmd)}\n")
!{' '.join(cmd)}

## 6. 📊 Hiển Thị Bảng Phân Rã Thời Gian (Khớp Table 9 & Table 2 Bài Báo Gốc)

In [ ]:
import os, glob, json
import pandas as pd
from tabulate import tabulate
from IPython.display import display, HTML

# Baseline tham chiếu chính thức từ bài báo gốc ICML 2026 (Table 9 & Table 2)
PAPER_BASELINES = {
    "sdv1.5_ddim50": {
        "name": "SD 1.5 (DDIM-50 / n=50)",
        "t_lookahead": 1.10,
        "t_reward": 0.65,
        "t_target": 3.51,
        "t_total": 5.26,
        "peak_vram": 8.90,
    },
    "sdv1.5_ddpm100": {
        "name": "SD 1.5 (DDPM-100 / n=50)",
        "t_lookahead": 1.10,
        "t_reward": 0.65,
        "t_target": 7.02,
        "t_total": 8.77,
        "peak_vram": 8.90,
    },
    "sdxl_dmd1": {
        "name": "SDXL (DMD-1 / n=100)",
        "t_lookahead": 4.30,
        "t_reward": 1.30,
        "t_target": 50.00,
        "t_total": 55.60,
        "peak_vram": 33.84,
    },
}

# Tìm file kết quả JSON mới nhất
json_files = glob.glob(f"results/benchmark_efficiency/efficiency_{SETTING}_{METHOD}_*.json")
if not json_files:
    json_files = glob.glob("results/benchmark_efficiency/*.json")

if json_files:
    latest_json = max(json_files, key=os.path.getmtime)
    with open(latest_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    ref = PAPER_BASELINES.get(SETTING, {})

    rows = [
        [
            f"Vanilla LiDAR (Paper)",
            f"{ref.get('t_lookahead', '-'):.2f}s",
            f"{ref.get('t_reward', '-'):.2f}s",
            f"{ref.get('t_target', '-'):.2f}s",
            f"{ref.get('t_total', '-'):.2f}s",
            f"{ref.get('peak_vram', '-'):.2f} GiB",
        ],
        [
            f"<b>{METHOD.upper()} (Measured)</b>",
            f"<b>{data['mean_t_lookahead_sec']:.2f}s</b>",
            f"<b>{data['mean_t_reward_sec']:.2f}s</b>",
            f"<b>{data['mean_t_target_sec']:.2f}s</b>",
            f"<b>{data['mean_t_total_sec']:.2f}s</b>",
            f"<b>{data['max_peak_vram_gib']:.2f} GiB</b>",
        ]
    ]

    headers = ["Method", "T_lookahead", "T_reward", "T_target", "T_total / prompt", "Peak VRAM"]
    df_summary = pd.DataFrame(rows, columns=headers)

    print(f"\n{'='*80}")
    print(f"📊 KẾT QUẢ ĐỐI CHIẾU TABLE 9 & TABLE 2 (Setting: {SETTING})")
    print(f"{'='*80}")
    print(tabulate(rows, headers=headers, tablefmt="grid"))
    print(f"{'='*80}\n")

    # Hiển thị bảng HTML rich
    display(HTML(f"<h3>📊 Bảng Đối Chiếu Thời Gian & Bộ Nhớ: {ref.get('name', SETTING)}</h3>"))
    display(HTML(df_summary.to_html(escape=False, index=False)))
else:
    print("⚠️ Chưa tìm thấy file kết quả trong 'results/benchmark_efficiency/'.")

## 7. 🔄 Chạy Tự Động So Sánh Trực Tiếp Cả 2 (Vanilla LiDAR vs RS-LiDAR)
> Cell này sẽ tự động chạy liên tiếp cả **Vanilla LiDAR** và **RS-LiDAR** trên cùng setting để xuất bảng so sánh sai khác $\Delta$ Time & Memory.

In [ ]:
# @title ⚡ Auto Comparative Run (LiDAR vs RS-LiDAR)
RUN_COMPARISON = False # @param {type:"boolean"}

if RUN_COMPARISON:
    print("\n🚀 BẮT ĐẦU CHẠY SO SÁNH 2 PHƯƠNG PHÁP...")
    
    # 1. Chạy Vanilla LiDAR
    print("\n--- [1/2] Đang chạy Vanilla LiDAR ---")
    !python benchmark_table2_efficiency.py --setting {SETTING} --method lidar --num_prompts {NUM_PROMPTS} --reward_batch_size None --warmup
    
    # 2. Chạy RS-LiDAR
    print("\n--- [2/2] Đang chạy RS-LiDAR ---")
    !python benchmark_table2_efficiency.py --setting {SETTING} --method rs-lidar --num_prompts {NUM_PROMPTS} --sigma {SIGMA} --num_mc_samples {NUM_MC_SAMPLES} --reward_batch_size None --warmup
    
    print("\n✅ Hoàn thành so sánh cả 2 phương pháp! Chạy lại Cell 6 để xem bảng tổng hợp đối chiếu.")
else:
    print("Đánh dấu tích vào RUN_COMPARISON để kích hoạt chạy so sánh tự động.")